# 02 — Retrieval Pipeline: BiomedBERT + FAISS

This notebook builds the retrieval component of our Medical RAG system:
1. Encode PubMed abstracts using BiomedBERT bi-encoder
2. Build a FAISS vector index
3. Test retrieval quality on sample queries

In [ ]:
# !pip install -q transformers datasets faiss-cpu sentence-transformers torch tqdm

In [ ]:
import sys
sys.path.append('..')

from src.data_utils import load_pubmedqa, filter_health_domain
from src.retriever import BiomedRetriever
import json
import time

## 1. Prepare the Retrieval Corpus

In [ ]:
# Load and filter data
artificial_ds = load_pubmedqa('pqa_artificial')
health_examples = filter_health_domain(artificial_ds['train'])
print(f"Health-domain examples for retrieval corpus: {len(health_examples)}")

# Extract abstracts as documents
documents = []
metadata = []
for ex in health_examples:
    if isinstance(ex.get('context'), dict):
        contexts = ex['context'].get('contexts', [])
        for ctx in contexts:
            documents.append(ctx)
            metadata.append({
                'pubid': ex.get('pubid', ''),
                'question': ex.get('question', ''),
            })

print(f"Total abstracts in corpus: {len(documents)}")
print(f"Average abstract length: {sum(len(d) for d in documents) / len(documents):.0f} chars")

## 2. Build the BiomedBERT Retriever

In [ ]:
# Initialize retriever with BiomedBERT
retriever = BiomedRetriever(
    model_name='microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
)
print(f"Model loaded on: {retriever.device}")

In [ ]:
# Build FAISS index (this takes a few minutes on T4)
# For Colab free tier, you may want to limit corpus size if memory is tight
MAX_DOCS = 10000  # Adjust based on available memory
corpus_subset = documents[:MAX_DOCS]
metadata_subset = metadata[:MAX_DOCS]

start = time.time()
retriever.build_index(corpus_subset, metadata_subset)
elapsed = time.time() - start
print(f"Index built in {elapsed:.1f}s")

In [ ]:
# Save index for reuse
retriever.save_index('../data/faiss_index.bin')
print("Index saved.")

## 3. Test Retrieval Quality

In [ ]:
# Test with sample queries
test_queries = [
    "Does metformin improve fertility in women with PCOS?",
    "Is there a link between endometriosis and infertility?",
    "Can hormonal contraceptives reduce menstrual pain?",
    "Does progesterone supplementation prevent miscarriage?",
    "Is estrogen therapy effective for menopausal symptoms?",
]

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    results = retriever.retrieve(query, top_k=3)
    for i, (doc, score, meta) in enumerate(results, 1):
        print(f"\n  [{i}] Score: {score:.4f}")
        print(f"      {doc[:200]}...")

## 4. Retrieval Metrics on Labeled Set

In [ ]:
# Evaluate retrieval: for each labeled example, check if the gold context is retrieved
from src.evaluation import compute_retrieval_metrics

labeled_ds = load_pubmedqa('pqa_labeled')
health_labeled = filter_health_domain(labeled_ds['train'])

# For each labeled example, the "gold" context is the abstract(s) paired with the question
# We check if retrieval finds text similar to the gold context
print(f"Evaluating retrieval on {len(health_labeled)} health-domain labeled examples...")
print("(This measures how well the retriever finds relevant abstracts)")

In [ ]:
# Compute embedding similarities for retrieval evaluation
import numpy as np

queries = [ex['question'] for ex in health_labeled]
query_embeddings = retriever.encode(queries)

# Get scores for top-K
K = 10
scores, indices = retriever.index.search(query_embeddings, K)

print(f"Average top-1 similarity: {scores[:, 0].mean():.4f}")
print(f"Average top-5 similarity: {scores[:, :5].mean():.4f}")
print(f"Average top-10 similarity: {scores.mean():.4f}")

## Next: Notebook 03 — Full RAG Pipeline

Now that retrieval is working, we'll add the BioMistral-7B generator to produce grounded answers.